# ಪಾಠ 03 - ಏಜೆಂಟಿಕ್ ವಿನ್ಯಾಸ ಮಾದರಿಗಳು

ಈ ಪಾಠದಲ್ಲಿ, ಪರಿಣಾಮಕಾರಿ AI ಏಜೆಂಟ್‌ಗಳನ್ನು ರೂಪಿಸಲು ಮೂರು ಮೂಲಭೂತ ವಿನ್ಯಾಸ ಮಾದರಿಗಳನ್ನು ಪರಿಶೀಲಿಸುತ್ತೇವೆ:

1. **ಸ್ಪಷ್ಟ ಏಜೆಂಟ್ ಸೂಚನೆಗಳು** — ಏಜೆಂಟ್ ವರ್ತನೆಯನ್ನು ಮಾರ್ಗದರ್ಶನ ಮಾಡುವ ನಿಖರ, ಪಾತ್ರನಿರ್ಬಂಧಿತ ಪ್ರಾಂಪ್ಟ್‌ಗಳನ್ನು ಸೃಷ್ಟಿಸುವುದು  
2. **ಪೈಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಗಳೊಂದಿಗೆ ರಚಿಸಲಾದ ಔಟ್‌ಪುಟ್** — ಏಜೆಂಟ್‌ಗಳು ನಿರೀಕ್ಷಿತ, ಪರಿಶೀಲಿತ ಡೇಟಾವನ್ನು ಹಿಂತಿರುಗಿಸುವ ವಿಷಯವನ್ನು ಖಚಿತಪಡಿಸುವುದು  
3. **ಒಂದು ಹೊಣೆಗಾರಿಕೆಯ ಏಜೆಂಟ್‌ ಗಳು** — ಪ್ರತಿ ಏಜೆಂಟ್ ಒಂದೇ ಕಛೇರಿ ಚೆನ್ನಾಗಿ ಮಾಡುವಂತೆ ವಿನ್ಯಾಸ ಮಾಡುವುದು  

ನಾವು ಪ್ರತಿ ಮಾದರಿಯನ್ನು **ಪ್ರವಾಸ ಗಂತব্য ಶಿಫಾರಸುಕಾರರು** ಪರಿಸ್ಥಿತಿಗೆ ಅನ್ವಯಿಸುತ್ತೇವೆ, ಹಂತದಿಂದ ಹಂತಕ್ಕೆ ಗಂತব্যಗಳನ್ನು ಶಿಫಾರಸು ಮಾಡುವುದು, ಲಭ್ಯತೆ ಪರಿಶೀಲಿಸುವುದು, ಮತ್ತು ಲಾಜಿಸ್ಟಿಕ್ಸ್ ನಿಭಾಯಿಸುವ ವ್ಯವಸ್ಥೆಯನ್ನು ನಿರ್ಮಿಸುತ್ತೇವೆ.


## ಸೆಟಪ್


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## ಮಾದರಿ 1: ಸ್ಪಷ್ಟ ಏಜೆಂಟ್ ಸೂಚನೆಗಳು

ಅತಿ ಪರಿಣಾಮಕಾರಿ ಮಾದರಿಯೂ ಸಹ ಸರಳವಾಗಿದೆ: ನಿಮ್ಮ ಏಜೆಂಟ್ ಗೆ ಸ್ಪಷ್ಟ, ವಿವರವಾದ ಸೂಚನೆಗಳನ್ನು ಬರೆಯುವುದು.

ಉತ್ತಮ ಸೂಚನೆಗಳು განსೂಚಿಸುತ್ತವೆ:
- ಏಜೆಂಟ್ ಯಾರು (ವ್ಯಕ್ತಿತ್ವ ಮತ್ತು ಶೈಲಿ)
- ಏನು ಮಾಡಬೇಕು (ಹೆಜ್ಜೆ-ಹೆಜ್ಜೆ ಜವಾಬ್ದಾರಿಗಳು)
- ಹೇಗೆ ವರಿಸಲು (ನಿಯಮಗಳು ಮತ್ತು ಶೈಲಿ)

ಕೆಳಗಿನಲ್ಲಿ, ನಾವು ಸ್ಪಷ್ಟ ಸೂಚನೆಗಳೊಂದಿಗೆ ಪ್ರವಾಸ ಸಲಹೆಗಾರ ಏಜೆಂಟ್ ರಚಿಸುತ್ತೇವೆ, ಇದು ಅದು ನೀಡುವ ಪ್ರತಿಯೊಂದು ಉತ್ತರವನ್ನು ರೂಪಿಸುತ್ತದೆ.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## ಮಾದರಿ 2: ಪೈಡಾಂಟಿಕ್ ಮಾದರಿಗಳೊಂದಿಗೆ ರಚಿಸಿರುವ	output

ಮುಕ್ತ ರೂಪದ ಪಠ್ಯ ಸಂಭಾಷಣೆಗೆ ಉಪಯುಕ್ತವಾಗಿದೆ, ಆದರೆ ಡೌನ್‌ಸ್ಟ್ರೀಮ್ ವ್ಯವಸ್ಥೆಗಳು ರಚಿಸಿದ ಡೇಟಾವನ್ನು ಅಗತ್ಯವಿದೆ.
**ಪೈಡಾಂಟಿಕ್ ಮಾದರಿಗಳ**ನ್ನು **ಟೂಲ್ ಫಂಕ್ಷನ್** ಜೊತೆಗೆ ಜೋಡಿಸುವ ಮೂಲಕ, ನಾವು:

- ಏಜೆಂಟ್‌ನ outputಗೆ ನಿಖರ ವಿನ್ಯಾಸವನ್ನು ನಿರ್ಧರಿಸಬಹುದು
- ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ಸ್ವಯಂಚಾಲಿತವಾಗಿ ಪರಿಶೀಲಿಸಬಹುದು
- ಏಜೆಂಟ್ ಫಲಿತಾಂಶಗಳನ್ನು ಅಪ್ಲಿಕೇಶನ್ ಲಾಜಿಕ್‌ನಲ್ಲಿ ನಂಬಬಹುದಾದ ರೀತಿಯಲ್ಲಿ ಎಣಿಸಬಹುದು

ನಾವು ಏಜೆಂಟ್ ತನ್ನ ಶಿಫಾರಸುಗಳನ್ನು ನೈಜ ಡೇಟಾದ ಆಧಾರದ ಮೇಲೆ ಸ್ಥಾಪಿಸಲು ಪ್ರಯಾಣ ಸ್ಥಳ ವಿವರಗಳನ್ನು ನೀಡುವ ಟೂಲ್ ಕೂಡ ಪರಿಚಯಿಸುತ್ತೇವೆ.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## ಮಾದರಿ 3: ಏಕ ಜವಾಬ್ದಾರಿಯ ಏಜೆಂಟುಗಳು

ಸಂಕೀರ್ಣ ಕೆಲಸಗಳು ಹಲವಾರು ಫೋಕಸ್ ಹೊಂದಿದ ಏಜೆಂಟ್‌ಗಳ ಮೂಲಕ ಹಂಚುವಿಕೆಯಿಂದ ಲಾಭ ಪಡೆಯುತ್ತವೆ, ಪ್ರತಿಯೊಂದಕ್ಕೆ ಒಂದು ಜವಾಬ್ದಾರಿ ಇರುತ್ತದೆ:

- ಸ್ಥಳಗಳು ಮತ್ತು ಲಭ್ಯತೆ ಬಗ್ಗೆ ತಿಳಿದಿರುವ **ಗಮ್ಯಸ್ಥಾನ ತಜ್ಞ**
- ವಿಮಾನಗಳು, ಹೋಟೆಲ್‌ಗಳು ಮತ್ತು ಪ್ರಯಾಣಕ್ರಮಗಳನ್ನು ನಿರ್ವಹಿಸುವ **ಲಾಗಿಸ್ಟಿಕ್ಸ್ ಯೋಜಕ**

ಇದು *ಆಶಯಗಳ ವಿಭಜನೆಯ* ಸಾಫ್ಟ್‌ವೇರ್ ಇಂಜಿನಿಯರಿಂಗ್ ತತ್ವದ ನಕಲು — ಪ್ರತಿ ಏಜೆಂಟ್ ಅನ್ನು ಸ್ವತಂತ್ರವಾಗಿ ಪರೀಕ್ಷಿಸಲು, ನಿರ್ವಹಿಸಲು ಮತ್ತು ಸುಧಾರಿಸಲು ಸುಲಭವಾಗಿದೆ.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## ಸಾರಾಂಗ

ಈ ಪಾಠದಲ್ಲಿ ನಾವು ಪ್ರವಾಸ ಶಿಫಾರಸು ಮಾಡುವ ಘಟಕಕ್ಕೆ ಮೂರು ಏಜೆಂಟಿಕ್ ವಿನ್ಯಾಸ ಮಾದರಿಗಳನ್ನು ಅನ್ವಯಿಸಿದ್ದೇವೆ:

| ಮಾದರಿ | ಪ್ರಮುಖ વિચાર | ಪ್ರಯೋಜನ |
|---|---|---|
| **ಸ್ಪಷ್ಟ ಸೂಚನೆಗಳು** | ವ್ಯಕ್ತಿತ್ವ, ಜವಾಬ್ದಾರಿಗಳು ಮತ್ತು ನಿರ್ಬಂಧಗಳನ್ನು ಮೊದಲಿನಲ್ಲೇ ನಿರ್ಧರಿಸುವುದು | ಸೀಮಿತ, ಬ್ರಾಂಡ್‌ಗೆ ಹೊಂದಿಕೊಳ್ಳುವ ಏಜೆಂಟ್ ವರ್ತನೆ |
| **ಸಂರಚಿತ ಉತ್ಪನ್ನ** | ಪ್ರತಿಕ್ರಿಯಾ ಸ್ವರೂಪವಾಗಿ ಪಿಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಗಳನ್ನು ಬಳಕೆ ಮಾಡುವುದು | ಪರಿಶೀಲಿತ, ಯಂತ್ರ ಓದಲು ಯೋಗ್ಯ ಫಲಿತಾಂಶಗಳು |
| **ಏಕ ಜವಾಬ್ದಾರಿ** | ಪ್ರತಿ ಏಜೆಂಟಿಗೆ ಒಂದು ಕೇಂದ್ರೀಕೃತ ಕೆಲಸ ನೀಡುವುದು | ಸುಲಭವಾಗಿ ಪರೀಕ್ಷೆ, ನಿರ್ವಹಣೆ ಮತ್ತು ಸಂಯೋಜನೆ ಸಾಧ್ಯ |

ಈ ಮಾದರಿಗಳು ಸಹಜವಾಗಿ ಸಂಯೋಜಿಸುತ್ತವೆ — ನೀವು ಸ್ಪಷ್ಟ ಸೂಚನೆಗಳನ್ನು ಸಂರಚಿತ ಉತ್ಪನ್ನದೊಂದಿಗೆ ಒಂದೇ ಜವಾಬ್ದಾರಿ ಏಜೆಂಟಿನ ಒಳಗೆ ಸಂಯೋಜಿಸಿ ಬಲಿಷ್ಠ, ಉತ್ಪಾದನೆ-ಸಿದ್ಧ ವ್ಯವಸ್ಥೆಗಳನ್ನು ನಿರ್ಮಿಸಬಹುದು.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅತ್ಯಾವಶ್ಯಕ ಸೂಚನೆ**:  
ಈ ಪಠ್ಯವನ್ನು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವದಿಸಲಾಗಿದೆ. ನಾವು ಶುದ್ಧತೆಯನ್ನು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ತಪ್ಪುಗಳು ಅಥವಾ ಅಸತ್ಯತೆಗಳು ಇರಬಹುದಾಗಿದೆ ಎಂದು ದಯವಿಟ್ಟು ಗಮನಿಸಿ. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜನ್ನು ಪ್ರಾಮಾಣಿಕೃತೆ ಹೊಂದಿರುವ ಮೂಲವಾಗಿ ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಸಲಹೆ ನೀಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದ ಬಳಕೆಯಿಂದ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪುಬರಹ ಅಥವಾ ತಪ್ಪು ಅರ್ಥಗ್ರಹಣಕ್ಕೆ ನಾವು ಜವಾಬ್ದಾರಿಯಾಗುವುದಿಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
